# 3번 — 정상 Reference + 누락 Vision Expert + LVLM 통합 Agent (dev240)

**기능 제작·CPU 검증 단계. 실제 새 통합 효과는 아직 미검증입니다.**

파일: `lvlm_selective_missing_expert_agent_dev_colab.ipynb`

L4/A100 선택 → 모두 실행 → Drive 연결 허용. **실제 추론 실행용**입니다. GitHub 토큰·추가 라벨링이 필요 없습니다.
기존 원본 ZIP 두 개를 자동 검색합니다. 1·2번을 다시 돌리거나 판별기를 다시 학습하지 않습니다.

- 고정된 dev240에서 새 Q2 유형 + Q3 9칸 복수 위치 첫 판단480호출. Q1은 기존 답 그대로/호출0.
- 유리 절연체에서 해당 질문의 Q2가 누락이 아니거나 Q3가 빈 위치일 때만 동결 DINOv2 누락 판별기 호출.
- 판별기 양성일 때만 같은 질문의 LVLM 확인1회 + 동일 사례 재질문 대조1회. 신규 LVLM480~680호출, DINO 최대50장(공유 캐시).
- 네 조건: LVLM 단독 / 판별기 양성의 Q2 단순 결합 / 도구 신호를 받고 재검토 / 같은 라우팅·횟수의 재질문.
- Q2·Q3는 서로 답을 전달하지 않습니다. 도구 음성으로 기존 누락/위치를 취소하지 않습니다. 도구는 위치를 출력하지 않습니다.
- 이 도구는 glass-insulator 정상/누락 라벨을 학습한 **지도학습 보조 조건**이며, 다른 자산/결함에 일반화했다고 주장하지 않습니다.
- 이미 분석한 dev입니다. 양성 위치 GT가 없어 위치 정확도는 null; Q3 유무/미탐·정상 오탐과 유형을 평가합니다. gold·재학습·임계값 튜닝0.
- 무효 출력은 원시 그대로 유지하고 오답으로 집계하며, 추론/도구 오류가 난 조건은 비교에서 제외합니다. ZIP 다운로드만으로 성공을 판단하지 마세요.
- 출력 폴더/코드/입력/모델이 같으면 중간 저장을 재사용합니다. 오류만 재시도하고 무효를 정답이 나올 때까지 다시 묻지 않습니다.

In [ ]:
N_IMAGES = 240
RUN_FOLDER = f"lvlm_insplad_results/selective_missing_expert_agent_dev{N_IMAGES}_v1"
SUP_ZIP_PATH = ""     # 자동 검색 실패 시에만 실제 Drive 경로
UNSUP_ZIP_PATH = ""

In [ ]:
import base64, hashlib, io, json, sys, zipfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive',force_remount=False)
DRIVE=Path('/content/drive/MyDrive')
SOURCE_SHA256='b9a5ba3e2d5e73a2780ecf2249d4c43526a7269de9191c8975d3763fbfc38852'
SOURCE_MANIFEST={'scripts/build_explainable_qa_gold.py': '9acdd42e87c15f9da2ba9bb7c3f89d9e77c42d774a8a8d38f6e783ea544d4f0a', 'scripts/build_gold_v2_dev_set.py': '31c1f17112eea9b279466cdeaa6fdafb43c450be2d6b268136f1b8b8a429f66a', 'scripts/build_retrieved_refs.py': '8f27f2aec2d7eecba622f651c2255a4da98ec435bbf9fd6fe2eedca96ddbed93', 'scripts/build_selective_missing_expert_agent_notebook.py': '597a36862282c6c388bbf22b8e277705b94259c7453c5e42ab54f8348aa6063c', 'scripts/dev_part_crop_annotation.py': '39de07b82f31778bfb0a94eac89265dd200296f7fa97e2c60ff6308b180682ea', 'scripts/loc_box_annotation.py': '1d839d6a6b10dda3c4c1d3c7ea16aa17e5d6b072f2df90d7e36c4d64fe316cff', 'scripts/q3_multiregion.py': '9eac922fd3ea6cb9164aa8ff8e511ad9a10e665da871e548192c3a53363a2063', 'scripts/run_dev_prompt_diagnostic.py': '76400860bc603002e6c3aa295b212b02bb2f78fd47b0d52f56c34ec2bd8dc8ab', 'scripts/run_gold_baselines.py': '978cb258c7412a767de2b5f7ee9604e726c29dc41922d656084d751d32869575', 'scripts/run_gold_v2_baselines.py': '72585045062bfc5f5ec5243ca50b34ba7fe4bb871f400bd734712354caec9a2a', 'scripts/run_gold_v2_consistency_metrics.py': '283ab1b1c2c1355895a0a047b9da5cfaef9ed67bf55f0108a0cfd6912e4f2dc2', 'scripts/run_gold_v2_dev_baselines.py': '809eca7a79e59f546107409f2a22f2bd1fda2cef3c77b8ea091b0cd401770fa9', 'scripts/run_gold_v2_q23_candidate.py': '19357ad3507dffd33ff5cd97d7febe36c87ab80a255e09398e380770c8d68499', 'scripts/run_gold_v2_q23_zoom_prompt_dev.py': 'cc8b7937fa75250b99af39ea5fa115cae06cb6f5aa470d0c28f417c4ed246d8a', 'scripts/run_gold_v2_q2_q3_agent_dev.py': '0dec0fd461b115de229913463ed7495552bcfb37a548dea0662f212800d695c8', 'scripts/run_gold_v2_q2_q3_refined_dev.py': '577837b328e7b1a9c8b0ea5a06638a798a9f043d84d58f7e085302b531ba8682', 'scripts/run_gold_v2_reference_ablation.py': 'fff6825808e7b9869c42ec2f54438cb98917c442fcdd2e6797d4554daaed9234', 'scripts/run_gold_v2_verify_agent.py': '0963c6c0c0e19c3bffd7fd7b68dcbd54ade6c51ba64bfd04f41fb0fafe5f966a', 'scripts/run_missing_component_probe.py': 'fcb36a2c74d5afaf2f9a1ffdd08a05660b37144cde8ed87d89ba55174b1adfc3', 'scripts/run_q2_conflict_evidence_dev.py': '52ccb22309b3771ef370bdfc271256c8eb39c603f6833900b72e6dd1a9f381c2', 'scripts/run_selective_missing_expert_agent_dev.py': '4cf263d5ffe7281237b99c943f2de528db9510a8a236e05017dd62d749dba5e2', 'src/insplad_benchmark/__init__.py': '5d9678c34324691fa8d7d5b507501945bb6eafd32e53ef6dedf1979e005cca49', 'src/insplad_benchmark/dataset.py': 'f724c3c5a76f4caa8fa86ac3d3fddef8fed0c466943f979b815f380fbd46652c', 'src/insplad_benchmark/hf_adapters.py': 'dbf178f1a6263c371fc8c7db192a33bdd252910b63267583d5bc24db11f684ab', 'src/insplad_benchmark/metrics.py': 'bb9386501a38031f3d5993744f02a4a1beeb3610bf2a1daa659100511f04cc9d', 'inputs/dataset.json': '8639fc3b991902b81548d9a975253ea7a3d51a15f7e1e29e478bffa9b9e6f527', 'inputs/frozen_detector.json': 'd9ff0610d0b74752a2dfc5b53f1cdc1db5e916e17b4260172f1c7abab8c6e7f1'}
SOURCE_BYTES=base64.b64decode('...')
assert hashlib.sha256(SOURCE_BYTES).hexdigest()==SOURCE_SHA256
CODE=Path('/content/selective_missing_expert_'+SOURCE_SHA256[:12])
with zipfile.ZipFile(io.BytesIO(SOURCE_BYTES)) as archive:
    assert set(archive.namelist())==set(SOURCE_MANIFEST)
    for name in archive.namelist():
        target=CODE/name
        assert target.resolve().is_relative_to(CODE.resolve())
        content=archive.read(name)
        assert hashlib.sha256(content).hexdigest()==SOURCE_MANIFEST[name]
        if target.exists() and target.read_bytes()!=content:
            raise RuntimeError('런타임 파일이 내장 코드와 다릅니다. 새 런타임을 사용하세요.')
        target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(content)
sys.path[:0]=[str(CODE/'scripts'),str(CODE/'src')]
import run_selective_missing_expert_agent_dev as experiment
assert N_IMAGES in (40,240), '기본240장. 작은 실행은40장/별도 폴더에서만 가능합니다.'
DATASET=CODE/'inputs/dataset.json'
DETECTOR=CODE/'inputs/frozen_detector.json'
assert hashlib.sha256(DETECTOR.read_bytes()).hexdigest()==experiment.DETECTOR_SHA256
OUT=DRIVE/RUN_FOLDER
assert OUT.name.startswith('selective_missing_expert_agent_dev')
print('실제 통합 실험 준비:',N_IMAGES,'장 / Q2·Q3 직접 질문',2*N_IMAGES,'회 + 필요한 검토/대조')
print('동결 판별기만 사용, 재학습0. SOURCE',SOURCE_SHA256)

In [ ]:
import subprocess
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory<20*1024**3:
    raise RuntimeError('L4/A100 VRAM20GB 이상 런타임을 선택하세요.')
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.14.0','accelerate==1.14.0',
    'sentencepiece==0.2.2','numpy>=1.26','pillow>=10','timm'],check=True)
def find_zip(name,size,override):
    if override:
        path=Path(override)
        if not path.is_file() or path.stat().st_size!=size:raise FileNotFoundError(str(path))
        return path
    for path in (DRIVE/name,DRIVE/'lvlm_insplad_data'/name):
        if path.is_file() and path.stat().st_size==size:return path
    matches=[p for p in DRIVE.rglob(name) if p.is_file() and p.stat().st_size==size]
    if not matches:raise FileNotFoundError(name)
    return sorted(matches)[0]
SUP=find_zip('supervised_fault_classification.zip',874753167,SUP_ZIP_PATH)
UNSUP=find_zip('unsupervised_anomaly_detection.zip',1170967719,UNSUP_ZIP_PATH)
from huggingface_hub import snapshot_download
snapshot=snapshot_download(experiment.MODEL_ID,revision=experiment.MODEL_REVISION)
# DINOv2 is loaded lazily only if the own-question route requests it.
command=[sys.executable,str(CODE/'scripts/run_selective_missing_expert_agent_dev.py'),
    '--dataset',str(DATASET),'--detector',str(DETECTOR),'--sup-zip',str(SUP),'--unsup-zip',str(UNSUP),
    '--model-path',snapshot,'--out-dir',str(OUT),'--n-images',str(N_IMAGES)]
LOG=DRIVE/f'selective_missing_expert_agent_dev{N_IMAGES}_console.log'
process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with LOG.open('a',encoding='utf-8') as log:
    for line in process.stdout:
        print(line,end='');log.write(line);log.flush()
RETURN_CODE=process.wait()
print('RUNNER_RETURN_CODE',RETURN_CODE)

In [ ]:
from google.colab import files
RESULT=DRIVE/f'selective_missing_expert_agent_dev{N_IMAGES}_results.zip'
with zipfile.ZipFile(RESULT,'w',zipfile.ZIP_DEFLATED) as archive:
    if OUT.exists():
        for path in sorted(OUT.rglob('*')):
            if path.is_file() and '_images' not in path.relative_to(OUT).parts:
                archive.write(path,'run/'+path.relative_to(OUT).as_posix())
    archive.writestr('source_bundle.zip',SOURCE_BYTES)
    archive.writestr('source_manifest.json',json.dumps(SOURCE_MANIFEST,indent=2))
    if LOG.exists():archive.write(LOG,'console.log')
    archive.writestr('runner_return_code.json',json.dumps({'return_code':RETURN_CODE}))
status_path=OUT/'run_status.json'
status=json.loads(status_path.read_text()) if status_path.exists() else {'complete':False,'status':'runner failed before status'}
print('RUN STATUS',status)
if RETURN_CODE!=0 or not status.get('complete'):
    print('오류가 난 조건은 비교 불가합니다. 원시 기록은 ZIP에 보존했습니다.')
elif status.get('invalid_lvlm_outputs'):
    print('평가 완료이나 무효 출력이 있습니다. 무효는 오답으로 집계했습니다:',status['invalid_lvlm_outputs'])
print('ZIP',RESULT,'SHA256',hashlib.sha256(RESULT.read_bytes()).hexdigest())
files.download(str(RESULT))